## SILVER LAYER
In Silver layer, I am applying data cleansing and transformation by defining the schema, enforcing consistent data types, and standardizing date formats. And also improving data quality by removing duplicates, handling null values, and filtering out invalid or negative records.

In [0]:
from pyspark.sql.functions import col, coalesce, lit, trim, when
# 1. READ
cust_raw = spark.readStream.table("az_adb_simbus_training.adarsh_training.bronze_customersraw_data")

# 2. DEFINING SCHEMA & STANDARDIZATION
cust_typed = cust_raw.select(
    col("c_custkey").cast("long"),
    col("c_name"),
    col("c_address"),
    col("c_nationkey").cast("long"),
    col("c_phone"),
    coalesce(col("c_acctbal").cast("decimal(18,2)"), lit(0.00)).alias("c_acctbal"),
    col("c_mktsegment"),
    col("c_comment")
)
# 3. FILTER & DEDUPLICATE
cust_clean = (cust_raw
    .select
    ( 
        col("c_custkey").cast("long"),
        #STRING CLEANING: Handle empty names or extra spaces
        trim(coalesce(col("c_name"), lit("UNKNOWN"))).alias("c_name"),
        col("c_address"),
        col("c_nationkey").cast("long"),
        col("c_phone"),
        #NEGATIVE VALUES: If balance is negative, treat as 0
        when(col("c_acctbal").cast("decimal(18,2)") < 0, lit(0.00))
        .otherwise(coalesce(col("c_acctbal").cast("decimal(18,2)"), lit(0.00)))
        .alias("c_acctbal"),
        col("c_mktsegment"),
        col("c_comment")
    )
    #INVALID DATA FILTER: Remove rows where critical info is missing
    .filter("""
        c_custkey IS NOT NULL 
        AND c_nationkey IS NOT NULL 
        AND c_name != 'UNKNOWN'
    """)
    .dropDuplicates(["c_custkey"])
)
 # 4. WRITE
(cust_clean.writeStream
    .option("checkpointLocation", "dbfs:/checkpoints/adarsh_training/silver_customers")
    .trigger(availableNow=True)
    .toTable("az_adb_simbus_training.adarsh_training.silver_customers")
)

In [0]:
from pyspark.sql.functions import col, upper, coalesce, lit, to_date, when
# 1. READ
orders_raw = spark.readStream.table("az_adb_simbus_training.adarsh_training.bronze_ordersraw_data")

# 2. DEFININ SCHEMA & MULTI-COLUMN CLEANING
ototalprice_numeric = col("o_totalprice").cast("decimal(18,2)")
orders_typed = orders_raw.select(
    col("o_orderkey").cast("long"),
    col("o_custkey").cast("long"),
    upper(coalesce(col("o_orderstatus"), lit("U"))).alias("o_orderstatus"),
    when(ototalprice_numeric < 0, lit(0.00))
        .otherwise(coalesce(ototalprice_numeric, lit(0.00)))
        .alias("o_totalprice"),
    to_date(col("o_orderdate")).alias("o_orderdate"),
    col("o_orderpriority"),
    col("o_clerk"),
    col("o_shippriority").cast("int"),
    col("o_comment"),
    when(col("o_orderkey").isNull(), lit("MISSING_ORDER_KEY"))
    .when(ototalprice_numeric < 0, lit("FIXED_NEGATIVE_PRICE"))
    .when(ototalprice_numeric.isNull(), lit("INVALID_PRICE_FORMAT"))
    .when(to_date(col("o_orderdate")).isNull(), lit("INVALID_DATE"))
    .otherwise(lit("VALID"))
    .alias("data_quality_flag")
)

orders_clean = orders_typed.filter("""
    o_orderkey IS NOT NULL 
    AND o_orderdate IS NOT NULL 
    AND o_custkey IS NOT NULL
""").dropDuplicates(["o_orderkey"])

(orders_clean.writeStream
    .option("checkpointLocation", "dbfs:/checkpoints/adarsh_training/silver_orders")
    .trigger(availableNow=True)
    .toTable("az_adb_simbus_training.adarsh_training.silver_orders"))

In [0]:
from pyspark.sql.functions import col, when, lit, coalesce, to_date

# 1. READ
lineitem_raw = spark.readStream.table("az_adb_simbus_training.adarsh_training.bronze_lineitemraw_data")

# 2. PRE-CAST NUMERICS (Fixes SparkNumberFormatException)
l_qty_decimal = col("l_quantity").cast("decimal(18,2)")
l_price_decimal = col("l_extendedprice").cast("decimal(18,2)")

# 3. DEFINING SCHEMA & CLEANING
lineitem_typed = lineitem_raw.select(
    col("l_orderkey").cast("long"),
    col("l_partkey").cast("long"),
    col("l_suppkey").cast("long"),
    col("l_linenumber").cast("int"),
    # Handle Negative/Malformed Quantities
    when(l_qty_decimal < 0, lit(0.00))
        .otherwise(coalesce(l_qty_decimal, lit(0.00)))
        .alias("l_quantity"),
    # Handle Malformed Prices
    coalesce(l_price_decimal, lit(0.00)).alias("l_extendedprice"),
    col("l_discount").cast("decimal(18,2)"),
    col("l_tax").cast("decimal(18,2)"),
    col("l_returnflag"),
    col("l_linestatus"),
    to_date(col("l_shipdate")).alias("l_shipdate"),
    to_date(col("l_commitdate")).alias("l_commitdate"),
    to_date(col("l_receiptdate")).alias("l_receiptdate"),
    col("l_shipinstruct"),
    col("l_shipmode"),
    col("l_comment"),
    # --- FLAG COLUMN ---
    when(col("l_orderkey").isNull(), lit("MISSING_ORDER_KEY"))
    .when(l_qty_decimal < 0, lit("FIXED_NEGATIVE_QTY"))
    .when(l_qty_decimal.isNull(), lit("INVALID_QTY_FORMAT"))
    .when(to_date(col("l_shipdate")).isNull(), lit("INVALID_SHIP_DATE"))
    .otherwise(lit("VALID"))
    .alias("data_quality_flag")
)

# 4. FILTER & WRITE
lineitem_clean = lineitem_typed.filter("l_orderkey IS NOT NULL") \
                               .dropDuplicates(["l_orderkey", "l_linenumber"])

(lineitem_clean.writeStream
    .option("checkpointLocation", "dbfs:/checkpoints/adarsh_training/silver_lineitem")
    .trigger(availableNow=True)
    .toTable("az_adb_simbus_training.adarsh_training.silver_lineitem"))

In [0]:
# 1. READ
nation_raw = spark.readStream.table("az_adb_simbus_training.adarsh_training.bronze_nationraw_data")

# 2. DEFININ SCHEMA & CLEANING
nation_typed = nation_raw.select(
    col("n_nationkey").cast("long"),
    upper(col("n_name")).alias("n_name"),
    col("n_regionkey").cast("long"),
    col("n_comment"),
    # --- FLAG COLUMN ---
    when(col("n_nationkey").isNull(), lit("MISSING_NATION_KEY"))
    .otherwise(lit("VALID"))
    .alias("data_quality_flag")
)
# 3. FILTER & WRITE
nation_clean = nation_typed.filter("n_nationkey IS NOT NULL") \
                           .dropDuplicates(["n_nationkey"])

(nation_clean.writeStream
    .option("checkpointLocation", "dbfs:/checkpoints/adarsh_training/silver_nation")
    .trigger(availableNow=True)
    .toTable("az_adb_simbus_training.adarsh_training.silver_nation"))

In [0]:
display(cust_clean)